这是**线性规划模型 (Linear Programming, LP)** 的详细解析。

它是优化类问题中**最基础、最常用**的模型。只要题目中出现“**资源有限**”、“**利润最大化**”、“**成本最小化**”，且变量之间是**按比例**变化的（没有平方、指数），首选线性规划。

---

### 一、 算法原理
**核心思想**：**“切蛋糕”与“爬山”。**

1.  **可行域（切蛋糕）**：所有的约束条件（比如钱不够、人手不够、时间有限）就像一把把刀，切掉不符合要求的区域。最后剩下的那个多边形区域，就是我们能选择的范围（可行域）。
2.  **目标函数（爬山）**：我们要在这个多边形区域里，找到一个点，让利润最高（山顶）或成本最低（谷底）。
3.  **单纯形法 (Simplex Method)**：数学证明告诉我们，**最优解一定在多边形的某个顶点（角）上**。算法只需要沿着多边形的边走，比较各个顶点的数值，就能找到最优解。

---

### 二、 推导与数学模型

标准形式（Scipy 求解器要求的形式）：

$$
\begin{aligned}
& \text{minimize} \quad z = c^T x \\
& \text{subject to} \quad A_{ub} x \le b_{ub} \quad \text{(不等式约束)} \\
& \quad \quad \quad \quad \quad A_{eq} x = b_{eq} \quad \text{(等式约束)} \\
& \quad \quad \quad \quad \quad l \le x \le u \quad \text{(变量边界)}
\end{aligned}
$$

**注意**：
*   **默认求最小值**：如果题目求最大值（Max），需要把目标函数系数加负号，变成求 $-z$ 的最小值。
*   **默认 $\le$ 约束**：如果题目是 $\ge$ 约束，两边同时乘以 $-1$ 变号。

---

### 三、 适用场景
1.  **生产计划**：原材料有限，生产A和B各多少个，利润最大？
2.  **配料问题**：在满足营养成分（维A、维B、蛋白质）的前提下，怎么搭配饲料成本最低？
3.  **运输问题**：从3个仓库运货到5个超市，运费最少怎么运？
4.  **投资组合**：风险不高于某值的情况下，收益最大。

---

### 四、 Python 代码框架

Python 中最标准的库是 `scipy.optimize.linprog`。
**这个代码框架是通用的**，你只需要把题目里的数字填进矩阵里。

```python
import numpy as np
from scipy.optimize import linprog

def solve_linear_programming():
    """
    线性规划求解模板
    题目示例：
    目标：Max Z = 4x1 + 3x2  (求最大利润)

    约束条件：
    1. 2x1 + x2 <= 10  (原料A限制)
    2. x1 + x2 <= 8    (原料B限制)
    3. x2 <= 7         (产能限制)
    4. x1, x2 >= 0     (非负限制)
    """

    # --- 1. 定义目标函数系数 c ---
    # 注意：linprog 默认求【最小值】(Min)。
    # 如果题目求【最大值】，必须对系数取反！
    # 原题 Max Z = 4x1 + 3x2  -->  Min -Z = -4x1 - 3x2
    c = [-4, -3]

    # --- 2. 定义不等式约束 (A_ub * x <= b_ub) ---
    # 将所有约束整理成 <= 的形式
    # 2x1 + 1x2 <= 10
    # 1x1 + 1x2 <= 8
    # 0x1 + 1x2 <= 7
    A_ub = np.array([
        [2, 1],
        [1, 1],
        [0, 1]
    ])
    b_ub = np.array([10, 8, 7])

    # --- 3. 定义等式约束 (A_eq * x = b_eq) ---
    # 如果没有等式约束，就留 None
    # 假设题目有个约束 x1 + x2 = 5，则：
    # A_eq = [[1, 1]]
    # b_eq = [5]
    A_eq = None
    b_eq = None

    # --- 4. 定义变量边界 (Bounds) ---
    # (min, max)，None代表无穷大
    # x1 >= 0 -> (0, None)
    # x2 >= 0 -> (0, None)
    x0_bounds = (0, None)
    x1_bounds = (0, None)

    # --- 5. 求解 ---
    # method='highs' 是目前 scipy 中性能最好的求解器
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                  bounds=[x0_bounds, x1_bounds], method='highs')

    # --- 6. 结果输出 ---
    if res.success:
        print("优化成功！")
        print(f"最优解 (x1, x2): {res.x}")
        # 记得把目标函数值取反回来（如果之前取反了的话）
        print(f"最大目标函数值 (Z): {-res.fun}")
    else:
        print("优化失败，原因:", res.message)

# ================= 使用示例 =================

if __name__ == "__main__":
    solve_linear_programming()
```

### 代码使用的“修修补补”指南：

1.  **怎么处理“大于等于” ($\ge$) 约束？**
    *   `linprog` 只认“小于等于” ($\le$)。
    *   如果题目有 $2x_1 + x_2 \ge 5$，请两边乘以 -1，变成：$-2x_1 - x_2 \le -5$。
    *   然后在 `A_ub` 里填 `[-2, -1]`，`b_ub` 里填 `-5`。

2.  **怎么处理“整数”约束？ (整数规划)**
    *   如果题目要求 $x_1, x_2$ 必须是**整数**（比如只能生产整辆车，不能生产半辆）。
    *   `scipy.optimize.linprog` **新版本 (1.9.0+)** 支持整数规划了！
    *   **修改方法**：在调用 `linprog` 时，加一个参数 `integrality`。
        ```python
        # 0表示连续变量，1表示整数变量
        # 假设 x1, x2 都必须是整数
        res = linprog(c, ..., integrality=[1, 1])
        ```
    *   如果你的 scipy 版本较老，你需要安装 `pulp` 库 (`pip install pulp`)，它写整数规划更简单。

3.  **变量太多，写矩阵会死人怎么办？**
    *   如果变量有几十上百个（比如运输问题 $i \times j$），手写 `A_ub` 矩阵是不可能的。
    *   **技巧**：用循环来构建矩阵，或者直接换用 `pulp` 库。
    *   在建模比赛中，如果问题规模小，用上面的 Scipy 模板；如果规模大且逻辑复杂，建议花10分钟学一下 `pulp`，它是像写英语句子一样写约束的。

4.  **结果显示 `None` 或 报错？**
    *   检查是不是约束冲突了（比如又要 $x>10$ 又要 $x<5$），导致**无解**。
    *   检查是不是没有边界限制（比如求最大值，但 $x$ 可以无限大），导致**无界解**。